# 02 · Leak & noise screen (Alert Triage #01)
Why the model keeps only a handful of native columns. NOISE = near-constant or redundant. LEAK = a column that alone almost perfectly separates the classes. Then a DOMINANCE screen keeps the gain balanced.

Thresholds come from `configs/feature_config.yaml`; the logic is `src/features/native_feature_screener.py`.

In [ ]:
import sys
from pathlib import Path
SLOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(SLOT))
from src.config import load_all_configs
from src import data_source as ds
cfg = load_all_configs()
mcfg, fcfg = cfg['model'], cfg['feature']
selected = ds.load_selected(mcfg)['features']
print('dataset', mcfg['data']['dataset'], '| model features', selected)


## The screen's own audit trail

In [ ]:
import pandas as pd
scr = pd.read_csv(SLOT/'data'/mcfg['data']['screen_audit'])
scr[['feature','family','role','gain_pct','single_feature_auc','zero_frac']]

## What was removed, and why

In [ ]:
sel = ds.load_selected(mcfg)
for reason, cols in sel['removed'].items():
    print(f'{reason}: {len(cols)}')
    for c in cols: print('   ', c)

## Single-feature AUC — the leak check
No kept feature may separate the classes on its own; that would be a data flaw the model memorises rather than behaviour it learns.

In [ ]:
floor = fcfg['screen']['leak']['single_feature_auc_min']
ax = scr.set_index('feature').single_feature_auc.sort_values().plot.barh(
        figsize=(8,8), color='#2a78d6')
ax.axvline(floor, ls='--', c='#d03b3b', label=f'leak floor {floor}'); ax.legend();
print('max single-feature AUC:', scr.single_feature_auc.max())

## Gain balance — no dominator, no dead weight

In [ ]:
bal = fcfg['screen']['balance']
g = sel['gain_pct']
print('cap', bal['gain_cap_pct'], '| max kept', max(g.values()))
print('floor', bal['gain_floor_pct'], '| min kept', min(g.values()))
g